# Evaluation Metrics (Qwen judge output)

This notebook computes the requested three metrics from:

- Output: `../output/test_main_eval_stream_batch_Qwen__Qwen3-Next-80B-A3B-Instruct-FP8.jsonl`

- Input reference set: `../input/combined_dataset_with_reference_good_row_idx.json`

Metrics:

1. **Agreement (mu ± std)**

   - Agreement with human/golden winner (`winner`) after mapping judge outputs to `{model_a, model_b, tie}`.

2. **Consistency (mu ± std)**

   - Multi-run consistency for the same question under same setting.

   - Uses **1-flip consistency** across repeats:

     - `1 - (# winner flips between adjacent repeats / (n-1))`

3. **Error Rate (mu ± std)**

   - Fraction of rows with wrong output format (invalid winner token or invalid single scores).

The final summary is aggregated over condition cells:

`judge_type x prompt_variant x temperature`.


In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)


def resolve_path(candidates: list[str]) -> Path:
    for cand in candidates:
        p = Path(cand)
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(f"Could not resolve any of: {candidates}")


OUTPUT_PATH = resolve_path([
    "output/results_raw_vllm/test_main_eval_stream_batch_google__gemma-3-27b-it.jsonl",
    "../output/results_raw_vllm/test_main_eval_stream_batch_google__gemma-3-27b-it.jsonl",
])

INPUT_PATH = resolve_path([
    "input/combined_dataset_with_reference_good_row_idx.json",
    "../input/combined_dataset_with_reference_good_row_idx.json",
])

MODEL_SLUG = OUTPUT_PATH.stem.replace("test_main_eval_stream_batch_", "")

print("Output:", OUTPUT_PATH)
print("Input :", INPUT_PATH)
print("Model :", MODEL_SLUG)


def _last_assistant_text(conv) -> str:
    if isinstance(conv, list):
        for msg in reversed(conv):
            if isinstance(msg, dict) and str(msg.get("role", "")).lower() == "assistant":
                return str(msg.get("content", "")).strip()
    return ""


# Output data (JSONL)
df_out = pd.read_json(OUTPUT_PATH, lines=True)

# Input reference data is JSONL despite .json suffix
df_ref = pd.read_json(INPUT_PATH, lines=True)

df_ref = df_ref.copy()
df_ref["answer_a_text"] = df_ref["conversation_a"].map(_last_assistant_text)
df_ref["answer_b_text"] = df_ref["conversation_b"].map(_last_assistant_text)
df_ref["gold_winner"] = df_ref["winner"]

merge_cols = ["row_idx", "dataset", "gold_winner", "reference_answer", "answer_a_text", "answer_b_text"]
df = df_out.merge(
    df_ref[merge_cols],
    how="left",
    left_on="question_id",
    right_on="row_idx",
)

print("\nLoaded output rows:", len(df_out))
print("Loaded input rows :", len(df_ref))
print("Merged rows       :", len(df))
print("Questions covered :", df["question_id"].nunique())
print("Judge types       :", df["judge_type"].value_counts().to_dict())
print("Prompt variants   :", df["prompt_variant"].value_counts().to_dict())
print("Temperatures      :", sorted(df["temperature"].dropna().unique().tolist()))
print("Repeats           :", sorted(df["repeat_id"].dropna().unique().tolist())[:3], "...", sorted(df["repeat_id"].dropna().unique().tolist())[-3:])
print("Datasets          :", df["dataset"].value_counts(dropna=False).to_dict())

missing_ref = int(df["reference_answer"].isna().sum())
missing_gold = int(df["gold_winner"].isna().sum())
print("Missing reference_answer rows:", missing_ref)
print("Missing gold_winner rows     :", missing_gold)

if missing_ref > 0:
    print("Sample question_id with missing reference:")
    display(df.loc[df["reference_answer"].isna(), ["dataset", "question_id"]].drop_duplicates().head(10))

Output: /home/snt/projects_lujun/LLMJudgeTempCausal/output/results_raw_vllm/test_main_eval_stream_batch_google__gemma-3-27b-it.jsonl
Input : /home/snt/projects_lujun/LLMJudgeTempCausal/input/combined_dataset_with_reference_good_row_idx.json
Model : google__gemma-3-27b-it

Loaded output rows: 180000
Loaded input rows : 500
Merged rows       : 180000
Questions covered : 500
Judge types       : {'pairwise': 60000, 'single_answer': 60000, 'reference_guided': 60000}
Prompt variants   : {'baseline': 90000, 'cot': 90000}
Temperatures      : [0.01, 0.5, 1.0, 1.5, 2.0, 3.0]
Repeats           : [0, 1, 2] ... [7, 8, 9]
Datasets          : {'mt_bench_human_judgments': 126000, 'mmlu_pro': 54000}
Missing reference_answer rows: 360
Missing gold_winner rows     : 0
Sample question_id with missing reference:


,dataset,question_id
439,mmlu_pro,439


In [2]:
# ---- Normalize judge outputs and compute core metrics ----

VALID_PAIRWISE = {"A", "B", "C"}


def map_pairwise_to_gold_label(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().upper()
    if s == "A":
        return "model_a"
    if s == "B":
        return "model_b"
    if s == "C":
        return "tie"
    return np.nan


# Ensure object dtype columns for mixed string/NaN values
if "predicted_winner" not in df.columns:
    df["predicted_winner"] = pd.Series(index=df.index, dtype="object")
if "format_error" not in df.columns:
    df["format_error"] = pd.Series(index=df.index, dtype="object")


# Pairwise/reference rows: token-based winner
pair_mask = df["judge_type"].isin(["pairwise", "reference_guided"])
df.loc[pair_mask, "winner_token"] = df.loc[pair_mask, "pairwise_winner"].astype(str).str.strip().str.upper()
df.loc[pair_mask, "format_error"] = ~df.loc[pair_mask, "winner_token"].isin(VALID_PAIRWISE)
df.loc[pair_mask, "predicted_winner"] = df.loc[pair_mask, "winner_token"].map(map_pairwise_to_gold_label)


# Single-answer rows: score-based winner
single_mask = df["judge_type"].eq("single_answer")
sa = pd.to_numeric(df.loc[single_mask, "score_a"], errors="coerce")
sb = pd.to_numeric(df.loc[single_mask, "score_b"], errors="coerce")
valid_single = sa.notna() & sb.notna() & sa.between(1, 10) & sb.between(1, 10)

single_pred = pd.Series(
    np.where(sa > sb, "model_a", np.where(sa < sb, "model_b", "tie")),
    index=df.index[single_mask],
    dtype="object",
)
single_pred.loc[~valid_single] = None

df.loc[single_mask, "predicted_winner"] = single_pred
df.loc[single_mask, "format_error"] = ~valid_single


# Safety fill for any unknown type
df["format_error"] = df["format_error"].fillna(True).astype(bool)


# Agreement with gold winner from input file
df["agreement"] = np.where(
    df["predicted_winner"].notna() & df["gold_winner"].notna(),
    (df["predicted_winner"] == df["gold_winner"]).astype(float),
    np.nan,
)


# 1-flip consistency helper

def one_flip_consistency(labels: list[str]) -> float:
    if len(labels) <= 1:
        return np.nan
    flips = sum(labels[i] != labels[i - 1] for i in range(1, len(labels)))
    return 1.0 - flips / (len(labels) - 1)


cons_rows = []
for (ds, jt, pv, temp, qid), g in df.groupby(
    ["dataset", "judge_type", "prompt_variant", "temperature", "question_id"],
    dropna=False,
):
    seq = g.sort_values("repeat_id")["predicted_winner"].dropna().astype(str).tolist()
    cons_rows.append(
        {
            "dataset": ds,
            "judge_type": jt,
            "prompt_variant": pv,
            "temperature": temp,
            "question_id": qid,
            "consistency_1flip": one_flip_consistency(seq),
        }
    )

df_cons = pd.DataFrame(cons_rows)


# Condition-level aggregates used for mu±std summary
cond_keys_with_repeat = ["dataset", "judge_type", "prompt_variant", "temperature", "repeat_id"]
cond_keys_no_repeat = ["dataset", "judge_type", "prompt_variant", "temperature"]

agreement_by_run = (
    df.groupby(cond_keys_with_repeat, dropna=False)["agreement"]
    .mean()
    .rename("agreement")
    .reset_index()
)

error_by_run = (
    df.groupby(cond_keys_with_repeat, dropna=False)["format_error"]
    .mean()
    .rename("error_rate")
    .reset_index()
)

consistency_by_cond = (
    df_cons.groupby(cond_keys_no_repeat, dropna=False)["consistency_1flip"]
    .mean()
    .rename("consistency")
    .reset_index()
)

print("Format-error overall:", round(df["format_error"].mean() * 100, 2), "%")
print("Agreement rows (non-null):", int(df["agreement"].notna().sum()))
print("Consistency rows (question-level):", int(df_cons["consistency_1flip"].notna().sum()))

print("\nAgreement by run (sample):")
display(agreement_by_run.head(8))

print("Error-rate by run (sample):")
display(error_by_run.head(8))

print("Consistency by condition (sample):")
display(consistency_by_cond.head(8))

Format-error overall: 0.65 %
Agreement rows (non-null): 178833
Consistency rows (question-level): 17997

Agreement by run (sample):


,dataset,judge_type,prompt_variant,temperature,repeat_id,agreement
0,mmlu_pro,pairwise,baseline,0.01,0,0.606667
1,mmlu_pro,pairwise,baseline,0.01,1,0.606667
2,mmlu_pro,pairwise,baseline,0.01,2,0.606667
3,mmlu_pro,pairwise,baseline,0.01,3,0.600000
4,mmlu_pro,pairwise,baseline,0.01,4,0.606667
5,mmlu_pro,pairwise,baseline,0.01,5,0.613333
6,mmlu_pro,pairwise,baseline,0.01,6,0.613333
7,mmlu_pro,pairwise,baseline,0.01,7,0.606667


Error-rate by run (sample):


,dataset,judge_type,prompt_variant,temperature,repeat_id,error_rate
0,mmlu_pro,pairwise,baseline,0.01,0,0.0
1,mmlu_pro,pairwise,baseline,0.01,1,0.0
2,mmlu_pro,pairwise,baseline,0.01,2,0.0
3,mmlu_pro,pairwise,baseline,0.01,3,0.0
4,mmlu_pro,pairwise,baseline,0.01,4,0.0
5,mmlu_pro,pairwise,baseline,0.01,5,0.0
6,mmlu_pro,pairwise,baseline,0.01,6,0.0
7,mmlu_pro,pairwise,baseline,0.01,7,0.0


Consistency by condition (sample):


,dataset,judge_type,prompt_variant,temperature,consistency
0,mmlu_pro,pairwise,baseline,0.01,0.991111
1,mmlu_pro,pairwise,baseline,0.50,0.986667
2,mmlu_pro,pairwise,baseline,1.00,0.977778
3,mmlu_pro,pairwise,baseline,1.50,0.971111
4,mmlu_pro,pairwise,baseline,2.00,0.963704
5,mmlu_pro,pairwise,baseline,3.00,0.901481
6,mmlu_pro,pairwise,cot,0.01,0.977778
7,mmlu_pro,pairwise,cot,0.50,0.955556


In [5]:
# ---- Test mode: keep only one question per (dataset, judge_type, prompt_variant, temperature) ----

TEST_ONE_QUESTION_PER_CONDITION = False

if "df_full_for_test" not in globals():
    df_full_for_test = df.copy()

base_df = df_full_for_test.copy()

if TEST_ONE_QUESTION_PER_CONDITION:
    cond_keys_no_repeat = ["dataset", "judge_type", "prompt_variant", "temperature"]
    cond_keys_with_repeat = ["dataset", "judge_type", "prompt_variant", "temperature", "repeat_id"]

    picked_questions = (
        base_df[cond_keys_no_repeat + ["question_id"]]
        .drop_duplicates()
        .sort_values(cond_keys_no_repeat + ["question_id"])
        .groupby(cond_keys_no_repeat, as_index=False)
        .first()
    )

    df = base_df.merge(
        picked_questions,
        on=cond_keys_no_repeat + ["question_id"],
        how="inner",
    ).copy()

    # Recompute consistency and run-level aggregates from sampled df
    cons_rows = []
    for (ds, jt, pv, temp, qid), g in df.groupby(
        ["dataset", "judge_type", "prompt_variant", "temperature", "question_id"],
        dropna=False,
    ):
        seq = g.sort_values("repeat_id")["predicted_winner"].dropna().astype(str).tolist()
        cons_rows.append(
            {
                "dataset": ds,
                "judge_type": jt,
                "prompt_variant": pv,
                "temperature": temp,
                "question_id": qid,
                "consistency_1flip": one_flip_consistency(seq),
            }
        )

    df_cons = pd.DataFrame(cons_rows)

    agreement_by_run = (
        df.groupby(cond_keys_with_repeat, dropna=False)["agreement"]
        .mean()
        .rename("agreement")
        .reset_index()
    )

    error_by_run = (
        df.groupby(cond_keys_with_repeat, dropna=False)["format_error"]
        .mean()
        .rename("error_rate")
        .reset_index()
    )

    consistency_by_cond = (
        df_cons.groupby(cond_keys_no_repeat, dropna=False)["consistency_1flip"]
        .mean()
        .rename("consistency")
        .reset_index()
    )

    print("[TEST MODE] One question kept per condition.")
    print("Picked conditions:", len(picked_questions))
    print("Sampled rows      :", len(df))
    print("Sampled questions :", df["question_id"].nunique())
    print("Dataset counts    :", df["dataset"].value_counts().to_dict())
    print("Prompt counts     :", df["prompt_variant"].value_counts().to_dict())
    print("Judge type counts :", df["judge_type"].value_counts().to_dict())
else:
    df = base_df.copy()
    print("[TEST MODE OFF] Using full dataset rows:", len(df))

[TEST MODE OFF] Using full dataset rows: 180000


In [ ]:
df.to_json(f"output/evaluation_with_metrics_{MODEL_SLUG}.jsonl", orient="records", lines=True)

In [34]:
print("Agreement by run sample:")
display(agreement_by_run.head(8))

print("Error rate by run sample:")
display(error_by_run.head(8))

print("Consistency by condition sample:")
display(consistency_by_cond.head(8))

Agreement by run sample:


,dataset,judge_type,prompt_variant,temperature,repeat_id,agreement
0,mmlu_pro,pairwise,baseline,0.01,0,0.606667
1,mmlu_pro,pairwise,baseline,0.01,1,0.606667
2,mmlu_pro,pairwise,baseline,0.01,2,0.606667
3,mmlu_pro,pairwise,baseline,0.01,3,0.600000
4,mmlu_pro,pairwise,baseline,0.01,4,0.606667
5,mmlu_pro,pairwise,baseline,0.01,5,0.613333
6,mmlu_pro,pairwise,baseline,0.01,6,0.613333
7,mmlu_pro,pairwise,baseline,0.01,7,0.606667


Error rate by run sample:


,dataset,judge_type,prompt_variant,temperature,repeat_id,error_rate
0,mmlu_pro,pairwise,baseline,0.01,0,0.0
1,mmlu_pro,pairwise,baseline,0.01,1,0.0
2,mmlu_pro,pairwise,baseline,0.01,2,0.0
3,mmlu_pro,pairwise,baseline,0.01,3,0.0
4,mmlu_pro,pairwise,baseline,0.01,4,0.0
5,mmlu_pro,pairwise,baseline,0.01,5,0.0
6,mmlu_pro,pairwise,baseline,0.01,6,0.0
7,mmlu_pro,pairwise,baseline,0.01,7,0.0


Consistency by condition sample:


,dataset,judge_type,prompt_variant,temperature,consistency
0,mmlu_pro,pairwise,baseline,0.01,0.991111
1,mmlu_pro,pairwise,baseline,0.50,0.986667
2,mmlu_pro,pairwise,baseline,1.00,0.977778
3,mmlu_pro,pairwise,baseline,1.50,0.971111
4,mmlu_pro,pairwise,baseline,2.00,0.963704
5,mmlu_pro,pairwise,baseline,3.00,0.901481
6,mmlu_pro,pairwise,cot,0.01,0.977778
7,mmlu_pro,pairwise,cot,0.50,0.955556


In [7]:
# ---- Final summary: mu ± std (overall + by dataset) ----

def mu_std(series: pd.Series) -> tuple[float, float]:
    s = pd.to_numeric(series, errors="coerce").dropna()
    return float(s.mean()), float(s.std(ddof=1))


def metric_rows(
    scope: str,
    dataset_label: str,
    a_series: pd.Series,
    c_series: pd.Series,
    e_series: pd.Series,
) -> list[dict]:
    a_mu, a_std = mu_std(a_series)
    c_mu, c_std = mu_std(c_series)
    e_mu, e_std = mu_std(e_series)
    return [
        {
            "scope": scope,
            "dataset": dataset_label,
            "Metric": "Agreement",
            "mu": a_mu,
            "std": a_std,
            "mu±std": f"{a_mu:.4f} ± {a_std:.4f}",
            "mu%±std%": f"{a_mu*100:.2f}% ± {a_std*100:.2f}%",
        },
        {
            "scope": scope,
            "dataset": dataset_label,
            "Metric": "Consistency (1-flip)",
            "mu": c_mu,
            "std": c_std,
            "mu±std": f"{c_mu:.4f} ± {c_std:.4f}",
            "mu%±std%": f"{c_mu*100:.2f}% ± {c_std*100:.2f}%",
        },
        {
            "scope": scope,
            "dataset": dataset_label,
            "Metric": "Error Rate",
            "mu": e_mu,
            "std": e_std,
            "mu±std": f"{e_mu:.4f} ± {e_std:.4f}",
            "mu%±std%": f"{e_mu*100:.2f}% ± {e_std*100:.2f}%",
        },
    ]


summary_rows = []
summary_rows += metric_rows(
    scope="overall",
    dataset_label="ALL",
    a_series=agreement_by_run["agreement"],
    c_series=consistency_by_cond["consistency"],
    e_series=error_by_run["error_rate"],
)

for ds in sorted(df["dataset"].dropna().astype(str).unique().tolist()):
    summary_rows += metric_rows(
        scope="by_dataset",
        dataset_label=ds,
        a_series=agreement_by_run.loc[agreement_by_run["dataset"] == ds, "agreement"],
        c_series=consistency_by_cond.loc[consistency_by_cond["dataset"] == ds, "consistency"],
        e_series=error_by_run.loc[error_by_run["dataset"] == ds, "error_rate"],
    )

summary = pd.DataFrame(summary_rows)
print("Requested metrics (mu ± std):")
display(summary[["scope", "dataset", "Metric", "mu±std", "mu%±std%"]])

group_cols = ["dataset", "judge_type", "prompt_variant", "temperature"]

agreement_by_cond = (
    agreement_by_run.groupby(group_cols, dropna=False)["agreement"]
    .mean()
    .reset_index()
)

error_by_cond = (
    error_by_run.groupby(group_cols, dropna=False)["error_rate"]
    .mean()
    .reset_index()
)

cond_table = (
    agreement_by_cond.merge(error_by_cond, on=group_cols, how="outer")
    .merge(consistency_by_cond, on=group_cols, how="outer")
    .sort_values(["dataset", "judge_type", "prompt_variant", "temperature"])
    .reset_index(drop=True)
)

print("\nCondition breakdown (by dataset):")
display(cond_table)

summary_path = OUTPUT_PATH.with_name(f"evaluation_summary_mu_std_{MODEL_SLUG}.jsonl")
summary_by_dataset_path = OUTPUT_PATH.with_name(f"evaluation_summary_mu_std_by_dataset_{MODEL_SLUG}.jsonl")
cond_path = OUTPUT_PATH.with_name(f"evaluation_by_condition_{MODEL_SLUG}.jsonl")

summary.to_json(summary_path, orient="records", lines=True)
summary.to_json(summary_by_dataset_path, orient="records", lines=True)
cond_table.to_json(cond_path, orient="records", lines=True)

print("\nSaved:")
print("-", summary_path)
print("-", summary_by_dataset_path)
print("-", cond_path)

Requested metrics (mu ± std):


,scope,dataset,Metric,mu±std,mu%±std%
0,overall,ALL,Agreement,0.5755 ± 0.0608,57.55% ± 6.08%
1,overall,ALL,Consistency (1-flip),0.8824 ± 0.1051,88.24% ± 10.51%
2,overall,ALL,Error Rate,0.0067 ± 0.0219,0.67% ± 2.19%
3,by_dataset,mmlu_pro,Agreement,0.6094 ± 0.0582,60.94% ± 5.82%
4,by_dataset,mmlu_pro,Consistency (1-flip),0.8657 ± 0.1147,86.57% ± 11.47%
5,by_dataset,mmlu_pro,Error Rate,0.0074 ± 0.0235,0.74% ± 2.35%
6,by_dataset,mt_bench_human_judgments,Agreement,0.5415 ± 0.0413,54.15% ± 4.13%
7,by_dataset,mt_bench_human_judgments,Consistency (1-flip),0.8990 ± 0.0931,89.90% ± 9.31%
8,by_dataset,mt_bench_human_judgments,Error Rate,0.0061 ± 0.0201,0.61% ± 2.01%



Condition breakdown (by dataset):


,dataset,judge_type,prompt_variant,temperature,agreement,error_rate,consistency
0,mmlu_pro,pairwise,baseline,0.01,0.607333,0.000000,0.991111
1,mmlu_pro,pairwise,baseline,0.50,0.605333,0.000000,0.986667
2,mmlu_pro,pairwise,baseline,1.00,0.604667,0.000000,0.977778
3,mmlu_pro,pairwise,baseline,1.50,0.605333,0.000000,0.971111
4,mmlu_pro,pairwise,baseline,2.00,0.604000,0.000000,0.963704
...,...,...,...,...,...,...,...
67,mt_bench_human_judgments,single_answer,cot,0.50,0.467908,0.002857,0.853550
68,mt_bench_human_judgments,single_answer,cot,1.00,0.476653,0.002571,0.823623
69,mt_bench_human_judgments,single_answer,cot,1.50,0.472508,0.002286,0.780317
70,mt_bench_human_judgments,single_answer,cot,2.00,0.479174,0.004286,0.709008



Saved:
- /home/snt/projects_lujun/LLMJudgeTempCausal/output/results_raw_vllm/evaluation_summary_mu_std_google__gemma-3-27b-it.jsonl
- /home/snt/projects_lujun/LLMJudgeTempCausal/output/results_raw_vllm/evaluation_summary_mu_std_by_dataset_google__gemma-3-27b-it.jsonl
- /home/snt/projects_lujun/LLMJudgeTempCausal/output/results_raw_vllm/evaluation_by_condition_google__gemma-3-27b-it.jsonl


In [9]:
# ---- Table like manuscript format: Human Bench vs MMLU_PRO ----

import numpy as np
import pandas as pd


def _fmt_mu_std(series: pd.Series, digits: int = 4) -> str:
    s = pd.to_numeric(series, errors="coerce").dropna()
    if len(s) == 0:
        return np.nan
    mu = float(s.mean())
    std = float(s.std(ddof=1)) if len(s) > 1 else 0.0
    return f"{mu:.{digits}f} ± {std:.{digits}f}"


def _short_model_name(x: str) -> str:
    if pd.isna(x):
        return x
    s = str(x)
    s = s.split("/")[-1]
    s = s.replace("-A3B-Instruct-FP8", "")
    s = s.replace("-Instruct", "")
    return s


def _one_flip_consistency(labels: list[str]) -> float:
    if len(labels) <= 1:
        return np.nan
    flips = sum(labels[i] != labels[i - 1] for i in range(1, len(labels)))
    return 1.0 - flips / (len(labels) - 1)


judge_type_map = {
    "pairwise": "Pairwise Comparison",
    "single_answer": "Single Answer Grading",
    "reference_guided": "Reference Guided Grading",
}
prompt_map = {"baseline": "Direct", "cot": "COT"}

run_group_cols = ["dataset", "judge_model", "judge_type", "prompt_variant", "temperature", "repeat_id"]

agreement_run_model = (
    df.groupby(run_group_cols, dropna=False)["agreement"]
    .mean()
    .rename("agreement")
    .reset_index()
)

error_run_model = (
    df.groupby(run_group_cols, dropna=False)["format_error"]
    .mean()
    .rename("error_rate")
    .reset_index()
)

cons_records = []
for key, g in df.groupby(["dataset", "judge_model", "judge_type", "prompt_variant", "temperature", "question_id"], dropna=False):
    seq = g.sort_values("repeat_id")["predicted_winner"].dropna().astype(str).tolist()
    cons_records.append(
        {
            "dataset": key[0],
            "judge_model": key[1],
            "judge_type": key[2],
            "prompt_variant": key[3],
            "temperature": key[4],
            "question_id": key[5],
            "consistency_1flip": _one_flip_consistency(seq),
        }
    )

cons_q_model = pd.DataFrame(cons_records)

cond_cols = ["dataset", "judge_model", "judge_type", "prompt_variant", "temperature"]
cond_keys = (
    df[cond_cols]
    .drop_duplicates()
    .sort_values(["dataset", "judge_model", "judge_type", "prompt_variant", "temperature"])
    .reset_index(drop=True)
)

rows = []
for r in cond_keys.itertuples(index=False):
    mask = (
        (agreement_run_model["dataset"] == r.dataset)
        & (agreement_run_model["judge_model"] == r.judge_model)
        & (agreement_run_model["judge_type"] == r.judge_type)
        & (agreement_run_model["prompt_variant"] == r.prompt_variant)
        & (agreement_run_model["temperature"] == r.temperature)
    )
    a_series = agreement_run_model.loc[mask, "agreement"]
    e_series = error_run_model.loc[mask, "error_rate"]
    c_mask = (
        (cons_q_model["dataset"] == r.dataset)
        & (cons_q_model["judge_model"] == r.judge_model)
        & (cons_q_model["judge_type"] == r.judge_type)
        & (cons_q_model["prompt_variant"] == r.prompt_variant)
        & (cons_q_model["temperature"] == r.temperature)
    )
    c_series = cons_q_model.loc[c_mask, "consistency_1flip"]
    rows.append(
        {
            "dataset": r.dataset,
            "T": r.temperature,
            "Model": _short_model_name(r.judge_model),
            "Judge Type": judge_type_map.get(r.judge_type, r.judge_type),
            "Prompt Strategy": prompt_map.get(r.prompt_variant, r.prompt_variant),
            "Agreement (µ ± std)": _fmt_mu_std(a_series, digits=2),
            "Consistency (µ ± std)": _fmt_mu_std(c_series, digits=2),
            "Error Rate (µ ± std)": _fmt_mu_std(e_series, digits=2),
        }
    )

stats_long = pd.DataFrame(rows)

datasets = stats_long["dataset"].dropna().astype(str).unique().tolist()
human_ds = next((d for d in datasets if "mt_bench" in d.lower() or "human" in d.lower()), datasets[0] if datasets else None)
mmlu_ds = next((d for d in datasets if "mmlu" in d.lower()), datasets[1] if len(datasets) > 1 else None)

if human_ds is None or mmlu_ds is None:
    raise ValueError(f"Need two datasets to build this table. Found: {datasets}")

base_cols = ["T", "Model", "Judge Type", "Prompt Strategy"]
metric_cols = [
    "Agreement (µ ± std)",
    "Consistency (µ ± std)",
    "Error Rate (µ ± std)",
]

human_part = stats_long.loc[stats_long["dataset"] == human_ds, base_cols + metric_cols].copy()
human_part = human_part.rename(columns={c: f"Human Bench {c}" for c in metric_cols})

mmlu_part = stats_long.loc[stats_long["dataset"] == mmlu_ds, base_cols + metric_cols].copy()
mmlu_part = mmlu_part.rename(columns={c: f"MMLU_PRO {c}" for c in metric_cols})

paper_table = (
    human_part.merge(mmlu_part, on=base_cols, how="outer")
    .sort_values(["T", "Model", "Judge Type", "Prompt Strategy"])
    .reset_index(drop=True)
)

print(f"Human Bench dataset: {human_ds}")
print(f"MMLU_PRO dataset: {mmlu_ds}")
display(paper_table)

paper_table_path = OUTPUT_PATH.with_name(f"evaluation_paper_style_table_{MODEL_SLUG}.jsonl")
paper_table.to_json(paper_table_path, orient="records", lines=True)
print("Saved:", paper_table_path)

Human Bench dataset: mt_bench_human_judgments
MMLU_PRO dataset: mmlu_pro


,T,Model,Judge Type,Prompt Strategy,Human Bench Agreement (µ ± std),Human Bench Consistency (µ ± std),Human Bench Error Rate (µ ± std),MMLU_PRO Agreement (µ ± std),MMLU_PRO Consistency (µ ± std),MMLU_PRO Error Rate (µ ± std)
0,0.01,gemma-3-27b-it,Pairwise Comparison,COT,0.56 ± 0.00,0.99 ± 0.08,0.00 ± 0.00,0.53 ± 0.01,0.98 ± 0.11,0.00 ± 0.00
1,0.01,gemma-3-27b-it,Pairwise Comparison,Direct,0.59 ± 0.00,1.00 ± 0.04,0.00 ± 0.00,0.61 ± 0.00,0.99 ± 0.08,0.00 ± 0.00
2,0.01,gemma-3-27b-it,Reference Guided Grading,COT,0.57 ± 0.01,0.96 ± 0.13,0.00 ± 0.00,0.68 ± 0.01,0.95 ± 0.15,0.00 ± 0.00
3,0.01,gemma-3-27b-it,Reference Guided Grading,Direct,0.57 ± 0.00,1.00 ± 0.04,0.00 ± 0.00,0.69 ± 0.01,0.99 ± 0.09,0.00 ± 0.00
4,0.01,gemma-3-27b-it,Single Answer Grading,COT,0.48 ± 0.01,0.94 ± 0.15,0.00 ± 0.00,0.64 ± 0.01,0.91 ± 0.18,0.00 ± 0.00
5,0.01,gemma-3-27b-it,Single Answer Grading,Direct,0.51 ± 0.00,0.98 ± 0.09,0.00 ± 0.00,0.56 ± 0.01,0.98 ± 0.11,0.00 ± 0.00
6,0.50,gemma-3-27b-it,Pairwise Comparison,COT,0.56 ± 0.00,0.97 ± 0.11,0.00 ± 0.00,0.54 ± 0.01,0.96 ± 0.14,0.00 ± 0.00
7,0.50,gemma-3-27b-it,Pairwise Comparison,Direct,0.59 ± 0.00,0.99 ± 0.07,0.00 ± 0.00,0.61 ± 0.00,0.99 ± 0.10,0.00 ± 0.00
8,0.50,gemma-3-27b-it,Reference Guided Grading,COT,0.56 ± 0.01,0.92 ± 0.19,0.00 ± 0.00,0.68 ± 0.02,0.90 ± 0.18,0.00 ± 0.00
9,0.50,gemma-3-27b-it,Reference Guided Grading,Direct,0.57 ± 0.00,0.99 ± 0.09,0.00 ± 0.00,0.68 ± 0.01,0.97 ± 0.14,0.00 ± 0.00


Saved: /home/snt/projects_lujun/LLMJudgeTempCausal/output/results_raw_vllm/evaluation_paper_style_table_google__gemma-3-27b-it.jsonl
